In [ ]:
!pip install langgraph langchain langchain_core langchain_groq python-dotenv typing langchain_google_genai

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

In [ ]:
class ChatState(TypedDict):

  messages: Annotated[list[BaseMessage], add_messages]

In [ ]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

def chat_node(state: ChatState):

    # take user query from state
    messages = state['messages']

    # send to llm
    response = llm.invoke(messages)

    # response store state
    return {'messages': [response]}

In [ ]:
checkpointer = MemorySaver()

graph = StateGraph(ChatState)

graph.add_node('chat_node', chat_node)

graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

chatbot = graph.compile(checkpointer=checkpointer)

In [ ]:
initial_state = {
    'messages': [HumanMessage(content='What is the capital of india')]
}

chatbot.invoke(initial_state)['messages']

In [ ]:
thread_id = '1'

while True:

  user_msg = input('Type here: ')
  print('User: ', user_msg)

  if user_msg.strip().lower() in ['exit', 'quit', 'bye']:
    break

  config = {'configurable': {'thread_id': thread_id}}
  response = chatbot.invoke({'messages': [HumanMessage(content=user_msg)]}, config=config)

  print('AI: ', response['messages'][-1].content)

In [ ]:
chatbot.get_state(config=config)